In [ ]:
# ============================================================
# LoRA + AdaLoRA + IA3 на FLUX.2 Klein (4D RoPE)
# ============================================================
import os, gc, re, warnings, logging, traceback
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, numpy as np, pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import clip
from diffusers import (
    Flux2KleinPipeline, Flux2Transformer2DModel,
    AutoencoderKLFlux2, FlowMatchEulerDiscreteScheduler,
)
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import (
    LoraConfig, AdaLoraConfig, IA3Config,
    get_peft_model, PeftModel,
)
from huggingface_hub import login
from google.colab import drive, userdata

warnings.filterwarnings("ignore")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

# ============================================================
# Проверка VRAM
# ============================================================
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"VRAM: free={free/1e9:.2f} GB / total={total/1e9:.2f} GB")
if free / 1e9 < 13.0:
    raise SystemExit("Мало VRAM. Runtime → Restart runtime.")

# ============================================================
# HF + Drive
# ============================================================
try:
    login(token=userdata.get("HF_TOKEN"))
    print(" Логин в HF")
except Exception:
    print(" HF_TOKEN не найден")

drive.mount('/content/drive')

# ============================================================
# Пути и гиперпараметры
# ============================================================
BASE_DIR    = "/content/drive/MyDrive/Colab Notebooks/LoraDataset"
OUTPUT_ROOT = "/content/drive/MyDrive/Colab Notebooks/flux_results"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
metadata_path = os.path.join(BASE_DIR, "metadata.csv")
MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"

IMAGE_SIZE = 256
MAX_STEPS  = 500
LR         = 2e-4
GRAD_ACCUM = 4
LIMIT_N    = 10

prompts = [
    "qwerty style, A work by an unknown artist from the 18th century. Oval, depicting a bouquet of flowers on a dark background",
    "qwerty style, A work by an unknown artist from the 18th century. Square, depicting a landscape with a tower on a rocky riverbank",
    "qwerty style, A work by an unknown artist from the 18th century. Rectangular, depicting a landscape with a waterfall in the foreground, a hut in the background",
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

# ============================================================
# Датасет
# ============================================================
df = pd.read_csv(metadata_path, encoding="utf-8")

def normalize_path(x):
    x = str(x).strip().strip('"').strip("'").replace("\\", "/")
    if os.path.isabs(x):
        return os.path.normpath(x)
    return os.path.normpath(os.path.join(BASE_DIR, x))

df["file_name"] = df["file_name"].apply(normalize_path)
df = df.head(LIMIT_N).reset_index(drop=True)
df = df[df["file_name"].apply(os.path.exists)].reset_index(drop=True)
print(f"Используем {len(df)} изображений")

class DS(Dataset):
    def __init__(self, df, tok, size=256):
        self.df, self.tok = df, tok
        self.t = transforms.Compose([
            transforms.Resize(size),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        px = self.t(Image.open(r["file_name"]).convert("RGB"))
        ids = self.tok(r["caption"], padding="max_length", max_length=512,
                       truncation=True, return_tensors="pt")["input_ids"].squeeze(0)
        return px, ids

ds = DS(df, tokenizer, size=IMAGE_SIZE)

# ============================================================
# 1. Предвычисление латентов (VAE)
# ============================================================
print("\n1. Предвычисление латентов...")
vae = AutoencoderKLFlux2.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.bfloat16)
vae.to("cuda")

all_latents = []
with torch.no_grad():
    for i in range(len(ds)):
        px, _ = ds[i]
        lat = vae.encode(px.unsqueeze(0).to("cuda", dtype=torch.bfloat16)).latent_dist.sample()
        all_latents.append(lat.to(torch.bfloat16).cpu())

vae.to("cpu"); del vae
gc.collect(); torch.cuda.empty_cache()
print(f"   латентов: {len(all_latents)}, форма: {all_latents[0].shape}")

# ============================================================
# 2. Предвычисление эмбеддингов (Qwen3 слои 9/18/27)
# ============================================================
print("\n2. Предвычисление эмбеддингов...")
text_encoder = AutoModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16),
    device_map="cuda", torch_dtype=torch.bfloat16,
).eval()

LAYERS = [9, 18, 27]
all_emb = []
with torch.no_grad():
    for i in range(len(ds)):
        _, ids = ds[i]
        out = text_encoder(ids.unsqueeze(0).to("cuda"), output_hidden_states=True)
        stacked = torch.cat([out.hidden_states[l] for l in LAYERS], dim=-1)  # [1, 512, 7680]
        all_emb.append(stacked.to(torch.bfloat16).cpu())

del text_encoder
gc.collect(); torch.cuda.empty_cache()
print(f"   эмбеддингов: {len(all_emb)}, форма: {all_emb[0].shape}")

# ============================================================
# 3. Patchify + 4D ids
# ============================================================
def patchify_latents(lat):
    # [B, C, H, W] → [B, (H/2)*(W/2), C*4]
    B, C, H, W = lat.shape
    lat = lat.view(B, C, H // 2, 2, W // 2, 2).permute(0, 2, 4, 1, 3, 5)
    return lat.reshape(B, (H // 2) * (W // 2), C * 4)

def make_img_ids(lat):
    # 4D RoPE: [B, (H/2)*(W/2), 4] = [t, row, col, extra]
    B, C, H, W = lat.shape
    h, w = H // 2, W // 2
    ids = torch.zeros(B, h * w, 4, device=lat.device, dtype=torch.float32)
    grid_y = torch.arange(h, device=lat.device).repeat_interleave(w)
    grid_x = torch.arange(w, device=lat.device).repeat(h)
    ids[:, :, 0] = 0.0        # t
    ids[:, :, 1] = grid_y     # row
    ids[:, :, 2] = grid_x     # col
    ids[:, :, 3] = 0.0        # extra
    return ids

def make_txt_ids(ehs):
    # 4D RoPE для текста: все нули
    return torch.zeros(ehs.shape[0], ehs.shape[1], 4,
                       device=ehs.device, dtype=torch.float32)

# Проверка форм
_l = all_latents[0].to("cuda")
_p = patchify_latents(_l)
_i = make_img_ids(_l)
print(f"\n patchify: {_l.shape} → {_p.shape}")
print(f" img_ids:  {_i.shape}")
assert _p.shape[-1] == 128, "patchify должен давать 128 каналов"
assert _i.shape[-1] == 4, "img_ids должен быть 4D"
del _l, _p, _i
gc.collect(); torch.cuda.empty_cache()

# ============================================================
# 4. Конфигурации PEFT
# ============================================================
TARGET_MODULES = ["to_q", "to_k", "to_v"]

configs = {
    "lora": LoraConfig(
        r=8, lora_alpha=8,
        target_modules=TARGET_MODULES,
        lora_dropout=0.1, bias="none",
    ),
    "adalora": AdaLoraConfig(
        r=8, lora_alpha=8,
        target_modules=TARGET_MODULES,
        lora_dropout=0.1, bias="none",
        total_step=MAX_STEPS, init_r=12, target_r=4,
        beta1=0.85, beta2=0.85, orth_reg_weight=0.5,
    ),
    "ia3": IA3Config(
        target_modules=TARGET_MODULES,
        feedforward_modules=[],
    ),
}
METHODS = ["lora", "adalora", "ia3"]

# ============================================================
# 5. Обучение
# ============================================================
def train_method(method):
    out_dir = os.path.join(OUTPUT_ROOT, f"{method}_results")
    os.makedirs(out_dir, exist_ok=True)

    gc.collect(); torch.cuda.empty_cache()
    print(f"\n{'='*60}\n {method.upper()}\n{'='*60}")

    train_tf = Flux2Transformer2DModel.from_pretrained(
        MODEL_ID, subfolder="transformer", torch_dtype=torch.bfloat16,
    )
    train_tf.to("cuda")
    train_tf.enable_gradient_checkpointing()

    model = get_peft_model(train_tf, configs[method])
    model.print_trainable_parameters()

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    model.train()
    step = 0
    try:
        while step < MAX_STEPS:
            for idx in range(len(all_latents)):
                latents = all_latents[idx].to("cuda")
                ehs = all_emb[idx].to("cuda")

                packed = patchify_latents(latents)        # [1, 256, 128]
                img_ids = make_img_ids(latents)           # [1, 256, 4]
                txt_ids = make_txt_ids(ehs)               # [1, 512, 4]

                # Flow matching
                noise = torch.randn_like(packed)
                ts = torch.rand((packed.shape[0],), device="cuda", dtype=torch.float32) * 1000.0
                t = (ts / 1000.0).to(torch.bfloat16)[:, None, None]
                noisy = (1 - t) * packed + t * noise
                target = noise - packed

                pred = model(
                    hidden_states=noisy,
                    encoder_hidden_states=ehs,
                    timestep=ts.to(torch.bfloat16),
                    img_ids=img_ids,
                    txt_ids=txt_ids,
                    guidance=None,
                    return_dict=False,
                )[0]

                loss = torch.nn.functional.mse_loss(pred.float(), target.float())
                loss.backward()

                if (step + 1) % GRAD_ACCUM == 0:
                    optimizer.step()
                    optimizer.zero_grad()

                if step % 20 == 0:
                    print(f"   step {step:>4}, loss: {loss.item():.4f}")
                step += 1
                if step >= MAX_STEPS:
                    break

        model.save_pretrained(out_dir)
        print(f" {method} сохранён: {out_dir}")
        ok = True
    except Exception as e:
        traceback.print_exc()
        print(f" {method} FAILED: {e}")
        ok = False

    del model, optimizer, train_tf
    gc.collect(); torch.cuda.empty_cache()
    return ok

trained = []
for m in METHODS:
    if train_method(m):
        trained.append(m)
print(f"\n Обучены: {trained}")

# ============================================================
# 6. Генерация
# ============================================================
print("\n" + "=" * 60 + "\nГЕНЕРАЦИЯ\n" + "=" * 60)

if trained:
    vae_gen = AutoencoderKLFlux2.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.bfloat16)
    vae_gen.to("cuda")

    text_encoder_gen = AutoModel.from_pretrained(
        MODEL_ID, subfolder="text_encoder",
        quantization_config=BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16),
        device_map="cuda", torch_dtype=torch.bfloat16,
    ).eval()

    for method in trained:
        adapter_dir = os.path.join(OUTPUT_ROOT, f"{method}_results")
        print(f"\n=== {method} ===")

        try:
            tf = Flux2Transformer2DModel.from_pretrained(
                MODEL_ID, subfolder="transformer", torch_dtype=torch.bfloat16,
            )
            tf = PeftModel.from_pretrained(tf, adapter_dir)
            tf = tf.merge_and_unload()
            tf.to("cuda")

            pipe = Flux2KleinPipeline(
                transformer=tf, vae=vae_gen, text_encoder=text_encoder_gen,
                tokenizer=tokenizer, scheduler=scheduler,
            )
            pipe.enable_attention_slicing()

            for i, p in enumerate(prompts):
                try:
                    img = pipe(prompt=p, height=512, width=512,
                               guidance_scale=1.0, num_inference_steps=4).images[0]
                except torch.cuda.OutOfMemoryError:
                    print(f"   {method} {i} OOM → 384x384")
                    torch.cuda.empty_cache()
                    img = pipe(prompt=p, height=384, width=384,
                               guidance_scale=1.0, num_inference_steps=4).images[0]
                img.save(os.path.join(OUTPUT_ROOT, f"{method}_result_{i}.png"))
                print(f"   saved {method} {i}")

            del pipe, tf
            gc.collect(); torch.cuda.empty_cache()
        except Exception as e:
            print(f"    {method}: {e}")
            traceback.print_exc()
            gc.collect(); torch.cuda.empty_cache()

    del vae_gen, text_encoder_gen
    gc.collect(); torch.cuda.empty_cache()

# ============================================================
# 7. CLIP + сетка
# ============================================================
print("\n" + "=" * 60 + "\nCLIP\n" + "=" * 60)

device = "cuda"
clip_model, preprocess = clip.load("ViT-B/32", device=device)

method_files = {}
for fname in os.listdir(OUTPUT_ROOT):
    m = re.match(r"(.+)_result_(\d+)\.png", fname)
    if m:
        method_files.setdefault(m.group(1), {})[int(m.group(2))] = os.path.join(OUTPUT_ROOT, fname)

methods_present = sorted(method_files.keys())
print(f"Найдены методы: {methods_present}")

if methods_present:
    num_prompts = max(len(v) for v in method_files.values())
    results = {}
    for method in methods_present:
        scores = []
        for i in range(num_prompts):
            fpath = method_files[method].get(i)
            if not fpath or not os.path.exists(fpath):
                scores.append(None); continue
            try:
                image = Image.open(fpath).convert("RGB")
                img_t = preprocess(image).unsqueeze(0).to(device)
                text = prompts[i] if i < len(prompts) else "qwerty style"
                txt_t = clip.tokenize([text]).to(device)
                with torch.no_grad():
                    fi = clip_model.encode_image(img_t); fi /= fi.norm(dim=1, keepdim=True)
                    ft = clip_model.encode_text(txt_t); ft /= ft.norm(dim=1, keepdim=True)
                    scores.append((fi @ ft.T).item())
            except Exception:
                scores.append(None)
        results[method] = scores

    print("\nCLIP Scores:")
    print("Method    | " + " | ".join([f"P{i}" for i in range(num_prompts)]) + " | Avg")
    for method in methods_present:
        s = results[method]
        valid = [x for x in s if x is not None]
        avg = np.mean(valid) if valid else 0
        sstr = " | ".join([f"{x:.4f}" if x is not None else "N/A" for x in s])
        print(f"{method:<9} | {sstr} | {avg:.4f}")

    rows, cols = len(methods_present), num_prompts
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    if rows == 1: axes = [axes]
    for r, method in enumerate(methods_present):
        for c in range(cols):
            fpath = method_files[method].get(c)
            if fpath and os.path.exists(fpath):
                axes[r][c].imshow(mpimg.imread(fpath))
                score = results[method][c] if results[method][c] is not None else 0.0
                axes[r][c].set_title(f"{method}\n{score:.3f}", fontsize=10)
            else:
                axes[r][c].text(0.5, 0.5, "N/A", ha="center", va="center")
            axes[r][c].axis("off")
    plt.tight_layout()
    grid_path = os.path.join(OUTPUT_ROOT, "comparison_grid.png")
    plt.savefig(grid_path, dpi=150)
    plt.show()
    print(f"\n Сетка: {grid_path}")

print(f"\n DONE: {OUTPUT_ROOT}")

In [ ]:
!pip uninstall -y diffusers peft transformers accelerate bitsandbytes xformers triton -q
!pip install git+https://github.com/huggingface/diffusers.git -q
!pip install git+https://github.com/huggingface/peft.git -q
!pip install transformers accelerate datasets bitsandbytes sentencepiece -q
!pip install --no-deps --upgrade "torchao>=0.17.0" -q
!pip install git+https://github.com/openai/CLIP.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 18.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, which is not installed.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 64.1 MB/s 